## Guidelines for Prompting
### In this lesson, you'll practice two prompting principles and their related tactics in order to write effective prompts for large language models.
## Setup
### Load the API key and relevant Python libaries.

In [16]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()
model="llama-3.1-8b-instant"
client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

#### helper function
Throughout this course, we will use Meta's `llama-3.1-8b-instant` model and the [chat completions endpoint](https://platform.openai.com/docs/guides/chat). 

This helper function will make it easier to use prompts and look at the generated outputs.  


In [17]:
from openai import OpenAI
def get_completion(prompt, model="llama-3.1-8b-instant"):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )

    return response.choices[0].message.content

**Note:** This and all other lab notebooks of this course use OpenAI library version `0.27.0`. 

In order to use the OpenAI library version `1.0.0`, here is the code that you would use instead for the `get_completion` function:

```python
client = openai.OpenAI()

def get_completion(prompt, model="llama-3.1-8b-instant"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content
```

## Prompting Principles
- **Principle 1: Write clear and specific instructions**
- **Principle 2: Give the model time to “think”**

### Tactics

#### Tactic 1: Use delimiters to clearly indicate distinct parts of the input
- Delimiters can be anything like: ```, """, < >, `<tag> </tag>`, `:`

In [18]:
text = f"""
You should express what you want a model to do by \ 
providing instructions that are as clear and \ 
specific as you can possibly make them. \ 
This will guide the model towards the desired output, \ 
and reduce the chances of receiving irrelevant \ 
or incorrect responses. Don't confuse writing a \ 
clear prompt with writing a short prompt. \ 
In many cases, longer prompts provide more clarity \ 
and context for the model, which can lead to \ 
more detailed and relevant outputs.
"""
prompt = f"""
Summarize the text delimited by triple backticks \ 
into a single sentence.
```{text}```
"""
response = get_completion(prompt)
print(response)

To get accurate and relevant responses from a model, provide clear and specific instructions in a prompt that is as detailed as possible.


#### Tactic 2: Ask for a structured output
- JSON, HTML

In [19]:
prompt = f"""
Generate a list of three made-up book titles along \ 
with their authors and genres. 
Provide them in JSON format with the following keys: 
book_id, title, author, genre.
"""
response = get_completion(prompt)
print(response)

Here are three made-up book titles in JSON format:

```json
[
  {
    "book_id": 1,
    "title": "The Echoes of Elyria",
    "author": "Ava Winters",
    "genre": "Fantasy"
  },
  {
    "book_id": 2,
    "title": "The Memory Weaver",
    "author": "Ethan Blackwood",
    "genre": "Science Fiction"
  },
  {
    "book_id": 3,
    "title": "The Shadowlands of Ashwood",
    "author": "Lila Flynn",
    "genre": "Mystery"
  }
]
```

Let me know if you'd like me to generate more!


#### Tactic 3: Ask the model to check whether conditions are satisfied

In [20]:
text_1 = f"""
Making a cup of tea is easy! First, you need to get some \ 
water boiling. While that's happening, \ 
grab a cup and put a tea bag in it. Once the water is \ 
hot enough, just pour it over the tea bag. \ 
Let it sit for a bit so the tea can steep. After a \ 
few minutes, take out the tea bag. If you \ 
like, you can add some sugar or milk to taste. \ 
And that's it! You've got yourself a delicious \ 
cup of tea to enjoy.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, \ 
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \ 
then simply write \"No steps provided.\"

\"\"\"{text_1}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 1:")
print(response)

Completion for Text 1:
Step 1 - Get some water boiling.
Step 2 - Grab a cup and put a tea bag in it.
Step 3 - Once the water is hot enough, pour it over the tea bag.
Step 4 - Let it sit for a bit so the tea can steep.
Step 5 - After a few minutes, take out the tea bag.
Step 6 - If you like, add some sugar or milk to taste.
Step 7 - Enjoy your delicious cup of tea.


In [21]:
text_2 = f"""
The sun is shining brightly today, and the birds are \
singing. It's a beautiful day to go for a \ 
walk in the park. The flowers are blooming, and the \ 
trees are swaying gently in the breeze. People \ 
are out and about, enjoying the lovely weather. \ 
Some are having picnics, while others are playing \ 
games or simply relaxing on the grass. It's a \ 
perfect day to spend time outdoors and appreciate the \ 
beauty of nature.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, \ 
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \ 
then simply write \"No steps provided.\"

\"\"\"{text_2}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 2:")
print(response)

Completion for Text 2:
No steps provided.


#### Tactic 4: "Few-shot" prompting

In [22]:
prompt = f"""
Your task is to answer in a consistent style.

<child>: Teach me about patience.

<grandparent>: The river that carves the deepest \ 
valley flows from a modest spring; the \ 
grandest symphony originates from a single note; \ 
the most intricate tapestry begins with a solitary thread.

<child>: Teach me about resilience.
"""
response = get_completion(prompt)
print(response)

<grandparent>: Ah, my dear child, the lotus flower blooms in the muddiest of waters, yet its petals remain unsoiled. It is a reminder that even in the darkest of times, we can rise above the challenges that surround us.


### Principle 2: Give the model time to “think” 

#### Tactic 1: Specify the steps required to complete a task

In [23]:
text = f"""
In a charming village, siblings Jack and Jill set out on \ 
a quest to fetch water from a hilltop \ 
well. As they climbed, singing joyfully, misfortune \ 
struck—Jack tripped on a stone and tumbled \ 
down the hill, with Jill following suit. \ 
Though slightly battered, the pair returned home to \ 
comforting embraces. Despite the mishap, \ 
their adventurous spirits remained undimmed, and they \ 
continued exploring with delight.
"""
# example 1
prompt_1 = f"""
Perform the following actions: 
1 - Summarize the following text delimited by triple \
backticks with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following \
keys: french_summary, num_names.

Separate your answers with line breaks.

Text:
```{text}```
"""
response = get_completion(prompt_1)
print("Completion for prompt 1:")
print(response)

Completion for prompt 1:
1 - The siblings Jack and Jill embark on a quest to fetch water from a hilltop well, but their journey is marred by a mishap.

2 - Les frères et sœurs Jack et Jill partent en quête d'eau d'un puits situé en haut d'une colline, mais leur voyage est entaché d'un malheur.

3 - 
- Jack
- Jill

4 - 
{
  "french_summary": "Les frères et sœurs Jack et Jill partent en quête d'eau d'un puits situé en haut d'une colline, mais leur voyage est entaché d'un malheur.",
  "num_names": 2
}


#### Ask for output in a specified format

In [24]:
prompt_2 = f"""
Your task is to perform the following actions: 
1 - Summarize the following text delimited by 
  <> with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the 
  following keys: french_summary, num_names.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of names in summary>
Output JSON: <json with summary and num_names>

Text: <{text}>
"""
response = get_completion(prompt_2)
print("\nCompletion for prompt 2:")
print(response)


Completion for prompt 2:
Text: <In a charming village, siblings Jack and Jill set out on \ a quest to fetch water from a hilltop \ well. As they climbed, singing joyfully, misfortune \ struck—Jack tripped on a stone and tumbled \ down the hill, with Jill following suit. \ Though slightly battered, the pair returned home to \ comforting embraces. Despite the mishap, \ their adventurous spirits remained undimmed, and they \ continued exploring with delight.>

Summary: Siblings Jack and Jill embarked on a quest to fetch water from a hilltop well but encountered misfortune.

Translation: Les frères et sœurs Jack et Jill ont entrepris une quête pour aller chercher de l'eau d'un puits situé en haut d'une colline mais ont rencontré du malheur.

Names: Jack, Jill

Output JSON:
{
  "french_summary": "Les frères et sœurs Jack et Jill ont entrepris une quête pour aller chercher de l'eau d'un puits situé en haut d'une colline mais ont rencontré du malheur.",
  "num_names": 2
}


#### Tactic 2: Instruct the model to work out its own solution before rushing to a conclusion

In [25]:
prompt = f"""
Determine if the student's solution is correct or not.

Question:
I'm building a solar power installation and I need \
 help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \ 
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations 
as a function of the number of square feet.

Student's Solution:
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
"""
response = get_completion(prompt)
print(response)

The student's solution is mostly correct, but there is a small mistake. 

The student correctly identified the costs:

1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 10x (not 100x)

The total cost should be the sum of these costs:

Total cost = 100x + 250x + 100,000 + 10x
Total cost = 360x + 100,000

So, the correct total cost for the first year of operations as a function of the number of square feet is 360x + 100,000.


#### Note that the student's solution is actually not correct.
#### We can fix this by instructing the model to work out its own solution first.

In [26]:
prompt = f"""
Your task is to determine if the student's solution \
is correct or not.
To solve the problem do the following:
- First, work out your own solution to the problem including the final total. 
- Then compare your solution to the student's solution \ 
and evaluate if the student's solution is correct or not. 
Don't decide if the student's solution is correct until 
you have done the problem yourself.

Use the following format:
Question:
```
question here
```
Student's solution:
```
student's solution here
```
Actual solution:
```
steps to work out the solution and your solution here
```
Is the student's solution the same as actual solution \
just calculated:
```
yes or no
```
Student grade:
```
correct or incorrect
```

Question:
```
I'm building a solar power installation and I need help \
working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations \
as a function of the number of square feet.
``` 
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
"""
response = get_completion(prompt)
print(response)

Question:
I'm building a solar power installation and I need help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost me a flat $100k per year, and an additional $10 / square foot
What is the total cost for the first year of operations as a function of the number of square feet.

Student's solution:
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000

Actual solution:
To find the total cost for the first year of operations, we need to add up the costs of the land, solar panels, and maintenance.

1. Land cost: The cost of the land is $100 per square foot, so the total cost of the land is 100x dollars.
2. Solar panel cost: The cost of the solar panels is $250 per square foot, so the total cost of the solar panels is 250x dollars.


## Model Limitations: Hallucinations
- Boie is a real company, the product name is not real.

In [27]:
prompt = f"""
Tell me about AeroGlide UltraSlim Smart Toothbrush by Boie
"""
response = get_completion(prompt)
print(response)

The AeroGlide UltraSlim Smart Toothbrush by Boie is a high-tech electric toothbrush designed for a seamless oral care experience. Here are some key features:

1. **Slim Design**: The UltraSlim toothbrush boasts a sleek and compact design, making it easy to maneuver in tight spaces and fit comfortably in your mouth.
2. **Smart Technology**: This toothbrush features advanced smart technology, including Bluetooth connectivity, allowing you to track your brushing habits, set reminders, and receive personalized feedback on your brushing technique.
3. **AeroGlide Technology**: The toothbrush's unique AeroGlide technology uses gentle, sweeping motions to clean teeth, ensuring a thorough and comfortable cleaning experience.
4. **Multiple Brushing Modes**: The UltraSlim toothbrush offers various brushing modes, including a gentle mode for sensitive teeth and gums, a normal mode for everyday cleaning, and a whitening mode for a brighter smile.
5. **Long-Lasting Battery**: The toothbrush is power